# TransJAX — Quickstart

**TransJAX** automatically translates Fortran scientific code into [JAX](https://github.com/google/jax) —
differentiable, GPU-ready Python — using a multi-agent LLM pipeline powered by Claude.

This notebook walks through the complete workflow:

1. [Installation & authentication](#1.-Installation-&-Authentication)
2. [Write a small Fortran module](#2.-Write-a-Fortran-Module)
3. [Static analysis with `FortranAnalyzer`](#3.-Static-Analysis)
4. [Translate to JAX via the Python API](#4.-Translate-to-JAX)
5. [Inspect the translated output](#5.-Inspect-the-Output)
6. [Run everything via the CLI](#6.-CLI-Quick-Reference)

> **Prerequisites**: Python ≥ 3.9, a Claude Pro/Max subscription **or** an Anthropic API key.

## 1. Installation & Authentication

In [20]:
import subprocess, sys, pathlib

repo_root = pathlib.Path.cwd().parent

# Ensure the src directory is on the path (works without kernel restart)
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Also install editable so CLI and dependencies are available
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(repo_root), "--quiet"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
else:
    print("transjax ready.")

transjax ready.


In [ ]:
!claude login

╭───ClaudeCodev2.1.72──────────────────────────────────────────────────────╮
││Tipsforgetting│
│WelcomebackAya!│started│
││Run/inittocreatea…│
│▐▛███▜▌│───────────────────────│
│▝▜█████▛▘│Recentactivity│
│▘▘▝▝│Norecentactivity│
│Sonnet4.6·ClaudePro·al4385@columbia.edu's││
│Organization││
│~/TransJAX/examples││
╰──────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────────────────────────────────────
❯  
────────────────────────────────────────────────────────────────────────────────
?forshortcuts
◐ medium · /effortlaude Code
❯                                                                           
────────────────────────────────────────────────────────────────────────────────
esctointerrupt◐medium·/effort
❯                                                                                
────────────────────────────────────────────────────────────────────────────────
esctointerrupt◐medium·/effort
✢h





✳h






### Authentication

TransJAX supports two authentication methods — pick whichever applies to you:

| Method | When to use | How to set up |
|--------|-------------|---------------|
| **Claude subscription** (Pro / Max) | You have an active Claude subscription | Run `claude login` in your terminal once |
| **Anthropic API key** | Pay-per-use API access | Set `ANTHROPIC_API_KEY` in your environment |

**Option 1 — Subscription login** (recommended if you have Claude Pro/Max):
```bash
# Run once in your terminal — stores credentials automatically
claude login
```
After running `claude login` the `CLAUDE_CODE_OAUTH_TOKEN` environment variable is set
automatically and TransJAX will pick it up — no further configuration needed.

**Option 2 — API key**:

In [11]:
import os

# --- Option 2: paste your API key here (skip if you used `claude login`) ---
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Verify that at least one auth method is present
has_oauth  = bool(os.environ.get("CLAUDE_CODE_OAUTH_TOKEN"))
has_apikey = bool(os.environ.get("ANTHROPIC_API_KEY"))

if has_oauth:
    print("✓ Authenticated via Claude subscription (claude login)")
elif has_apikey:
    print("✓ Authenticated via ANTHROPIC_API_KEY")
else:
    print("✗ No authentication found.")
    print("  Run `claude login` in your terminal, or set ANTHROPIC_API_KEY.")

✗ No authentication found.
  Run `claude login` in your terminal, or set ANTHROPIC_API_KEY.


---
## 2. Write a Fortran Module

We'll use a self-contained Fortran module that implements a few operations
common in numerical / scientific computing:
- a simple dot product
- a tridiagonal matrix-vector multiply
- one step of explicit Euler integration for a heat equation

In [ ]:
from pathlib import Path

# Create a temporary directory to hold our example Fortran source
# TransJAX templates expect source files inside a 'src/' subdirectory
fortran_dir = Path("./example_fortran")
(fortran_dir / "src").mkdir(parents=True, exist_ok=True)

fortran_source = '''\
! heat_numerics.f90
! Simple 1-D heat-equation numerics used to demonstrate TransJAX.

module heat_numerics
  implicit none

  ! Physical / numerical parameters
  real, parameter :: DEFAULT_ALPHA = 0.01   ! thermal diffusivity
  real, parameter :: DEFAULT_DX    = 0.1    ! grid spacing
  real, parameter :: DEFAULT_DT    = 0.001  ! time step

  contains

  ! ------------------------------------------------------------------
  ! dot_product_1d  — dot product of two real vectors
  ! ------------------------------------------------------------------
  function dot_product_1d(a, b, n) result(s)
    integer, intent(in) :: n
    real,    intent(in) :: a(n), b(n)
    real :: s
    integer :: i
    s = 0.0
    do i = 1, n
      s = s + a(i) * b(i)
    end do
  end function dot_product_1d

  ! ------------------------------------------------------------------
  ! tridiag_matvec  — multiply a tridiagonal matrix by a vector
  !   lower(:)  sub-diagonal  (indices 2..n)
  !   diag(:)   main diagonal (indices 1..n)
  !   upper(:)  super-diagonal(indices 1..n-1)
  ! ------------------------------------------------------------------
  subroutine tridiag_matvec(lower, diag, upper, x, y, n)
    integer, intent(in)  :: n
    real,    intent(in)  :: lower(n), diag(n), upper(n), x(n)
    real,    intent(out) :: y(n)
    integer :: i

    y(1) = diag(1)*x(1) + upper(1)*x(2)
    do i = 2, n-1
      y(i) = lower(i)*x(i-1) + diag(i)*x(i) + upper(i)*x(i+1)
    end do
    y(n) = lower(n)*x(n-1) + diag(n)*x(n)
  end subroutine tridiag_matvec

  ! ------------------------------------------------------------------
  ! euler_step  — one explicit-Euler step of the 1-D heat equation
  !   u_new(i) = u(i) + alpha * dt/dx^2 * (u(i+1) - 2*u(i) + u(i-1))
  ! ------------------------------------------------------------------
  subroutine euler_step(u, u_new, n, alpha, dx, dt)
    integer, intent(in)  :: n
    real,    intent(in)  :: u(n), alpha, dx, dt
    real,    intent(out) :: u_new(n)
    real    :: r
    integer :: i

    r = alpha * dt / (dx * dx)
    u_new(1) = u(1)  ! fixed left boundary
    do i = 2, n-1
      u_new(i) = u(i) + r * (u(i+1) - 2.0*u(i) + u(i-1))
    end do
    u_new(n) = u(n)  ! fixed right boundary
  end subroutine euler_step

end module heat_numerics
'''
(fortran_dir / "src" / "heat_numerics.f90").write_text(fortran_source)
print(f"Wrote Fortran source to: {fortran_dir / 'src' / 'heat_numerics.f90'}")
print(f"Wrote Fortran source to: {fortran_dir / 'heat_numerics.f90'}")

Wrote Fortran source to: example_fortran/heat_numerics.f90


---
## 3. Static Analysis

`FortranAnalyzer` (no LLM required) parses source files, builds a dependency
graph, and identifies the correct translation order.
It is useful for inspecting a codebase **before** committing to translation.

In [19]:
from transjax import quick_analyze

stats = quick_analyze(str(fortran_dir))

print("=== Static Analysis Summary ===")
for key, value in stats.items():
    print(f"  {key:<30} {value}")

[10:26:08] INFO     Creating configuration using template: generic                            ]8;id=855446;file:///Users/ayalahlou/TransJAX/src/transjax/analyzer/config/project_config.py\project_config.py]8;;\:]8;id=456000;file:///Users/ayalahlou/TransJAX/src/transjax/analyzer/config/project_config.py#310\310]8;;\

           ERROR    Configuration error: None of the source directories exist:                ]8;id=22637;file:///Users/ayalahlou/TransJAX/src/transjax/analyzer/config/project_config.py\project_config.py]8;;\:]8;id=717982;file:///Users/ayalahlou/TransJAX/src/transjax/analyzer/config/project_config.py#160\160]8;;\
                    ['/Users/ayalahlou/TransJAX/examples/example_fortran/src']                                     

ValueError: Invalid configuration: /Users/ayalahlou/TransJAX/examples/example_fortran does not exist or is not accessible

In [ ]:
from transjax import create_analyzer_for_project

analysis_dir = "./example_analysis"

analyzer = create_analyzer_for_project(
    str(fortran_dir),
    template="generic",          # auto-detect project type, or choose: ctsm, scientific_computing
    output_dir=analysis_dir,
    generate_graphs=False,        # set True to save dependency-graph images
)
analyzer.analyze(save_results=True)

print("Analysis saved to:", analysis_dir)

In [ ]:
import json

# Load and pretty-print the JSON report
report_path = Path(analysis_dir) / "analysis_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2)[:2000], "...")
else:
    print("Report not found — check the analysis_dir path above.")

---
## 4. Translate to JAX

`OrchestratorAgent` runs the full pipeline:
1. Static analysis → translation order
2. `TranslatorAgent` converts each module (calls Claude)
3. `TestAgent` generates pytest tests for the translated code
4. `RepairAgent` iteratively fixes any failing tests

**This step makes LLM API calls** and requires valid authentication (see Section 1).

In [ ]:
from transjax import OrchestratorAgent

output_dir = Path("./example_jax_output")

orchestrator = OrchestratorAgent(
    fortran_dir=fortran_dir,
    output_dir=output_dir,
    # model="claude-sonnet-4-5",   # default; change to claude-opus-4-6 for harder code
    # temperature=0.0,             # deterministic output
    max_repair_iterations=3,       # retry limit if generated tests fail
    skip_tests=False,              # generate + run pytest tests
    skip_repair=False,             # enable the repair loop
    verbose=True,
)

results = orchestrator.run()

print("\n=== Translation Results ===")
print(f"  Modules translated : {results['translated_count']}")
print(f"  Tests passed       : {results['tests_passed']}")
print(f"  Final failures     : {results['final_failures']}")

In [ ]:
# Cost summary (shows $0.00 for subscription users; token counts are always shown)
cost = orchestrator.get_cost_estimate()
print("=== Token / Cost Summary ===")
for k, v in cost.items():
    print(f"  {k:<25} {v}")

---
## 5. Inspect the Output

After a successful run the output directory has the following layout:

```
example_jax_output/
├── src/                     ← translated JAX modules
│   └── heat_numerics.py
├── tests/                   ← auto-generated pytest files
│   └── test_heat_numerics.py
├── docs/                    ← translation notes per module
└── reports/
    └── translation_summary.json
```

In [ ]:
import os

print("Output directory tree:")
for root, dirs, files in os.walk(output_dir):
    dirs.sort()
    level = len(Path(root).relative_to(output_dir).parts)
    indent = "    " * level
    print(f"{indent}{Path(root).name}/")
    for f in sorted(files):
        print(f"{indent}    {f}")

In [ ]:
# Show the translated JAX source
translated = output_dir / "src" / "heat_numerics.py"
if translated.exists():
    print(translated.read_text())
else:
    # Translation may have failed — check the report
    print("Translated file not found. Check reports/translation_summary.json")

In [ ]:
# Show the auto-generated tests
test_file = output_dir / "tests" / "test_heat_numerics.py"
if test_file.exists():
    print(test_file.read_text())

In [ ]:
# Full translation report
summary = output_dir / "reports" / "translation_summary.json"
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))

---
## 6. CLI Quick Reference

Everything above can also be driven from the command line:

```bash
# One-time auth setup (subscription users)
claude login

# Or: one-time env file setup (API key users)
transjax init          # creates .env.template
cp .env.template .env  # fill in ANTHROPIC_API_KEY

# Inspect the codebase without translating
transjax analyze ./example_fortran

# Full pipeline: translate + test + repair
transjax convert ./example_fortran -o ./example_jax_output

# Useful flags
transjax convert ./example_fortran \
    --model claude-opus-4-6 \
    --max-repair-iterations 5 \
    --modules heat_numerics \
    --force           # re-translate even if output already exists
    --skip-tests      # skip test generation

# Show the current configuration
transjax show-config
```

---
## Using Individual Agents

You can also call each agent directly for more control:

In [ ]:
# Example: translate a single module without running the full pipeline
from transjax import TranslatorAgent

translator = TranslatorAgent()

# Read the Fortran source
fortran_code = (fortran_dir / "src" / "heat_numerics.f90").read_text()

result = translator.translate_module(
    module_name="heat_numerics",
    fortran_code=fortran_code,
    source_dir=str(fortran_dir),
)

print("--- Translated physics module ---")
print(result.physics_code[:1500])

In [ ]:
# Example: generate tests for an already-translated module
from transjax import TestAgent

tester = TestAgent()

test_result = tester.generate_tests(
    module_name="heat_numerics",
    fortran_code=fortran_code,
    jax_code=result.physics_code,
)

print("--- Generated test file ---")
print(test_result.test_code[:1500])

---
## Next Steps

- Try **your own Fortran codebase**: point `fortran_dir` at any directory of `.f90` / `.f` files.
- Tune the model with `--model claude-opus-4-6` for more complex scientific code.
- Read the [TransJAX README](../README.md) for advanced configuration (custom YAML config,
  project templates, cost caps, etc.).
- File bugs or feature requests at the project repository.